In [ ]:
!pip install wandb -qU

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision.datasets import CIFAR10
import pandas as pd
import numpy as np
import math
import tqdm
import wandb

In [ ]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 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


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saitejaalasyam (saitejaalasyam-arizona-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
train = CIFAR10(root='./data', train=True, download=True)
test = CIFAR10(root='./data', train=False, download=True)

100%|██████████| 170M/170M [07:36<00:00, 373kB/s]


In [ ]:
type(train.data)

numpy.ndarray

In [ ]:
device = torch.accelerator.current_accelerator()
device

device(type='cuda')

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, dataset, transform = None, target_transform = None):
    self.dataset = dataset
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.dataset.targets)

  def __getitem__(self, index):
    return self.dataset.data[index],self.dataset.targets[index]

In [ ]:
traindataset = CustomDataset(train)
testdataset = CustomDataset(test)


In [ ]:
class CustomDataLoader:

  def __init__(self,dataset: CustomDataset,batch_size, shuffle):
    self.dataset = dataset
    self.batch_size = batch_size
    self.shuffle = shuffle

  def __iter__(self):

    indices = torch.arange(len(self.dataset))
    if self.shuffle:
      indices = indices[torch.randperm(len(self.dataset))]

    for i in range(0, len(self.dataset),self.batch_size):

      batch_indices = indices[i : i+self.batch_size]
      batch = [self.dataset[idx.item()] for idx in batch_indices]

      batch = self.collate_fn(batch)
      yield batch
  def __len__(self):
    return math.ceil(len(self.dataset)/self.batch_size)
  def collate_fn(self,batch):

    xs, ys = zip(*batch)
    xs = [torch.tensor(x, dtype = torch.float32) / 255.0 for x in xs]
    return torch.stack(xs).permute(0,3,1,2), torch.tensor(ys)


In [ ]:
train_dataloader = CustomDataLoader(traindataset, batch_size=32, shuffle=True)
test_dataloader = CustomDataLoader(testdataset, batch_size=32, shuffle=False)

In [ ]:
def relu(x):
  return torch.max(x, torch.zeros_like(x))

def softmax(x, dim = -1):
  x_max = torch.max(x,dim = dim, keepdim = True)[0]
  rescaled = x - x_max
  exp_rescaled = torch.exp(rescaled)
  exp_sum = torch.sum(exp_rescaled, dim = dim, keepdim = True)
  return exp_rescaled / exp_sum

def log_softmax(x,dim = -1):
  x_max = torch.max(x,dim = dim, keepdim = True)[0]
  rescaled = x - x_max
  exp_rescaled = torch.exp(rescaled)
  exp_sum = torch.sum(exp_rescaled, dim = dim, keepdim = True)
  return rescaled - torch.log(exp_sum)

def crossentropyloss(inputs,targets):
  negative_logits = -log_softmax(inputs)
  return torch.mean(torch.gather(negative_logits, -1, targets.unsqueeze(-1)))

def lr_scheduler(it, warmup_it, cosine_it, min_learning, max_learning):

  if it < warmup_it:
    return max_learning* (it/warmup_it)

  if it > cosine_it:
    return min_learning

  decay = (it- warmup_it)/(cosine_it - warmup_it)
  decay_rate = 0.5*(1 + math.cos(math.pi * decay))
  return decay_rate*(max_learning - min_learning) + min_learning

def clip_gradient(parameters, max_norm):

  grads = [p.grad for p in parameters if p.grad is not None]

  norm = torch.tensor(0.0,device = grads[0].device)

  for g in grads:

    norm += torch.sum(g**2)

  norm = torch.sqrt(norm)

  clip_coeff = min(1, max_norm/(norm+1e-6))

  for p in parameters:
    if p.grad is not None:
      p.grad.data.mul_(clip_coeff)


In [ ]:
from typing import Callable
from __future__ import annotations
class AdamW(torch.optim.Optimizer):

  def __init__(self, params,lr = 1e-3,betas = (0.9,0.999),eps = 1e-8,weight_decay = 0.01):
    defaults = dict(lr = lr,betas = betas, eps =eps, weight_decay= weight_decay)
    super().__init__(params,defaults)
  def step(self, closure : Callable | None = None):

    loss = None

    if closure is not None:
      loss = closure()

    for group in self.param_groups:
      for p in group['params']:
        if p.grad.data is None:
          continue
        state= self.state[p]
        grad = p.grad.data
        if len(state) ==0 :
          state['t'] = 1
          state['m'] = torch.zeros_like(grad)
          state['v'] = torch.zeros_like(grad)

        alpha= group['lr']
        beta1,beta2 = group['betas']
        eps = group['eps']
        weight_decay = group['weight_decay']

        t = state.get('t')
        prev_mt = state.get('m')
        prev_vt = state.get('v')

        alpha_t = alpha*(math.sqrt(1-beta2**t)/(1-beta1**t))
        p.data -= alpha * weight_decay * p.data

        m_t = prev_mt* beta1 + (1-beta1)*grad
        v_t = prev_vt * beta2 + (1-beta2)*(grad**2)

        p.data -= alpha_t * (m_t/(torch.sqrt(v_t)+eps))


        state['t'] = t + 1
        state['m'] = m_t
        state['v'] = v_t

    return loss


In [ ]:
def im2col(x, kernel_size, padding = 0, stride = 1):

  return torch.nn.functional.unfold(x, kernel_size=kernel_size, padding=padding, stride=stride)


In [ ]:
class Conv2d(torch.nn.Module):
  def __init__(self,in_channels,out_channels,kernel_size,padding =0,stride = 1 ):
    super().__init__()
    self.in_channels = in_channels
    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.padding = padding
    self.stride = stride

    # Scale initialization to prevent exploding variance
    scale = math.sqrt(in_channels * kernel_size * kernel_size)
    self.weight = torch.nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size, requires_grad = True) / scale)
    self.bias = torch.nn.Parameter(torch.zeros(out_channels))


  def forward(self,x):

    batch_size, num_channels, H, W = x.shape
    x_col = im2col(x,self.kernel_size, self.padding,self.stride)
    W_col = self.weight.view(self.out_channels, -1)


    out = torch.matmul(W_col,x_col) + self.bias.view(1,-1,1)

    H_out = (H + 2*self.padding - self.kernel_size)//self.stride + 1
    W_out = (W + 2*self.padding - self.kernel_size)//self.stride + 1


    out = out.view(batch_size, self.out_channels, H_out,W_out)


    return out


In [ ]:
class MaxPooling(torch.nn.Module):

  def __init__(self,kernel_size, stride):
    super().__init__()
    self.kernel_size = kernel_size
    self.stride = stride

  def forward(self,x):
    return torch.nn.functional.max_pool2d(x, self.kernel_size, self.stride)


In [ ]:
class Linear(torch.nn.Module):

  def __init__(self, in_features, out_features):
    super().__init__()
    scale = math.sqrt(in_features)
    self.weight = torch.nn.Parameter(torch.randn(out_features,in_features, requires_grad = True) / scale)
    self.bias = torch.nn.Parameter(torch.zeros(out_features, requires_grad = True))

  def forward(self,x):
    return x @ self.weight.t() + self.bias


In [ ]:
def LayerNorm(x,dim = -1,eps = 1e-5):

  mean = torch.mean(x,dim = dim, keepdim = True)
  var = torch.var(x,dim = dim, unbiased = False, keepdim = True)

  return (x-mean)/torch.sqrt(var + eps)

def RMSNorm(x,dim = -1, eps = 1e-5):

  denom = torch.sqrt((x**2).mean(dim = dim, keepdim = True)+eps)
  return x/denom

def BatchNorm(x,dim = 0):
  mean = torch.mean(x,dim = dim, keepdim = True)
  var = torch.var(x,dim = dim, unbiased = False, keepdim = True)

  return (x-mean)/torch.sqrt(var + 1e-5)

In [ ]:
class CNN(torch.nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = Conv2d(3,32,3,padding = 1).to(device)
    self.layer1 = BatchNorm
    self.relu1 = relu
    self.conv2 = Conv2d(32,32,3,padding = 1).to(device)
    self.layer2 = BatchNorm
    self.relu2 = relu
    self.pool1 = MaxPooling(2,2).to(device)


    self.conv3 = Conv2d(32,64,3,padding = 1).to(device)
    self.layer3 = BatchNorm
    self.relu3 = relu
    self.conv4 = Conv2d(64,64,3,padding = 1).to(device)
    self.layer4 = BatchNorm
    self.relu4 = relu
    self.pool2 = MaxPooling(2,2).to(device)
    self.Linear = Linear(4096, 10).to(device)

  def forward(self,x):

    x = self.conv1(x)
    x = self.layer1(x)
    x = self.relu1(x)
    x = self.conv2(x)
    x = self.layer2(x)
    x = self.relu2(x)
    x = self.pool1(x)
    x = self.conv3(x)
    x = self.layer3(x)
    x = self.relu3(x)
    x = self.conv4(x)
    x = self.layer4(x)
    x = self.relu4(x)
    x = self.pool2(x)
    x = x.view(x.shape[0],-1)

    x = self.Linear(x)
    return x


1563

In [ ]:
from torch.autograd import grad
import types

model = CNN()
optimizer = AdamW(model.parameters(),lr = 0.0001)
model = model.to(device)
warmup_it = 200
cosine_it = 15000//4
min_learning = 0.0001
max_learning = 0.01
epochs = 20
wandb.init(
    project=  "CIFAR10-CNN",
    config = {
        "epochs" : epochs,
        "batch_size" : 32,
        "architecture" : "CNN"
    }
)
def train(model,dataloader,optimizer,num_epochs,grad_accumulation_steps):
  global_step = 0
  for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for i,batch in enumerate(tqdm.tqdm(train_dataloader)):
      x,y = batch
      x = x.to(device)
      y = y.to(device)
      outputs = model(x)
      loss = crossentropyloss(outputs,y)

      scaled_loss = loss / grad_accumulation_steps
      scaled_loss.backward()


      total_loss += scaled_loss.item()
      if (i + 1) % grad_accumulation_steps == 0 or (i + 1) == len(dataloader):
        lr = lr_scheduler(global_step,warmup_it,cosine_it,min_learning,max_learning)

        for param_group in optimizer.param_groups:
          param_group['lr'] = lr
        clip_gradient(model.parameters(),1)
        optimizer.step()
        optimizer.zero_grad()
        wandb.log({
            "batch_loss":scaled_loss.item(),
            "learning_rate":lr,
            "global_step":global_step
        })

        global_step += 1
    print(f"Epoch: {epoch+1}, Loss: {total_loss/len(dataloader)}")

In [ ]:
train(model,train_dataloader,optimizer, 20,4)

100%|██████████| 1563/1563 [00:18<00:00, 85.16it/s]


Epoch: 1, Loss: 0.44878818419829286


100%|██████████| 1563/1563 [00:17<00:00, 89.60it/s]


Epoch: 2, Loss: 0.3500580705096915


100%|██████████| 1563/1563 [00:18<00:00, 86.59it/s]


...

Epoch: 18, Loss: 0.14493369965343925


100%|██████████| 1563/1563 [00:17<00:00, 88.69it/s]


Epoch: 19, Loss: 0.14529583944449878


100%|██████████| 1563/1563 [00:18<00:00, 86.43it/s]

Epoch: 20, Loss: 0.14501989179994537


In [ ]:
def evaluate(model,dataloader):
  predictions = []
  accuracy = 0
  with torch.no_grad():
    model.eval()
    for batch in tqdm.tqdm(dataloader):
      x,y = batch
      x = x.to(device)
      y = y.to(device)
      outputs = model(x)
      outputs = softmax(outputs)
      prediction = torch.argmax(outputs,dim = -1)

      predictions.append(prediction)
      accuracy += torch.sum(prediction == y).item()
  return accuracy/len(dataloader.dataset), predictions



In [ ]:
accuracy_score, predictions= evaluate(model, test_dataloader)
wandb.log({"test_accuracy":accuracy_score})
print(f"Accuracy: {accuracy_score}")

100%|██████████| 313/313 [00:01<00:00, 214.41it/s]

Accuracy: 0.7529
